# Parquet & PyArrow

## What is Parquet?

Parquet is a binary file format designed for analytical workloads. Unlike CSV which stores data row by row, Parquet stores data **column by column**. This has two major consequences:

- If you only need 3 columns out of 50, Parquet reads only those 3. CSV reads everything.
- Each column is compressed independently. Numbers compress better with numbers, strings with strings. This gives Parquet much better compression ratios than CSV.

---

## File Structure

A Parquet file is divided into **row groups** horizontally. Each row group contains a chunk of rows, and within each row group, data is stored column by column.

```
parquet file
├── Row Group 0  (rows 0 - 999)
│   ├── Column chunk: age
│   ├── Column chunk: price
│   └── Column chunk: sqft
├── Row Group 1  (rows 1000 - 1999)
│   ├── Column chunk: age
│   ├── Column chunk: price
│   └── Column chunk: sqft
└── Row Group 2  (rows 2000 - 2622)
    ├── Column chunk: age
    ├── Column chunk: price
    └── Column chunk: sqft
```

Row group size is decided at write time. The default in pandas is around 128MB per group.

---

## Why Row Groups Matter for Lazy Loading

Parquet does not support single-row random access. The smallest unit you can read is one row group. So when you want row 1500, you read the entire row group that contains row 1500, then index into it locally.

This is why lazy loading Parquet is not the same as lazy loading individual images — you always pay for a full chunk, not a single sample.

---

## Key PyArrow Concepts

### `pq.ParquetFile(path)`
Opens the file and reads only the **metadata** (schema, row count, row group boundaries). No data is loaded into memory.

### `pf.metadata`
The metadata object. Contains overall file info.

- `pf.metadata.num_rows` — total row count across the file
- `pf.metadata.num_row_groups` — how many row groups exist
- `pf.metadata.row_group(i).num_rows` — row count of group i specifically

### `pf.read_row_group(i)`
Reads row group i from disk into memory. Returns a `pyarrow.Table`.

### `pf.read()`
Reads the entire file into memory. Equivalent to `pq.read_table(path)`.

### `pyarrow.Table`
PyArrow's in-memory tabular format. Columnar. Can be converted with:
- `.to_pandas()` — returns a DataFrame
- `.to_pydict()` — returns a dict of lists
- `.column("col_name")` — returns a single column as a `ChunkedArray`

### `pq.read_table(path, columns=[...])`
Reads the whole file but only the specified columns. Efficient for column subsetting.

### `pq.read_metadata(path)`
Reads only metadata without opening a `ParquetFile` object. Lightweight.

---

## PyArrow Code Reference

```python
import pyarrow.parquet as pq
import pyarrow as pa

# --- Writing ---
import pandas as pd

df = pd.read_csv("data.csv")

# default: pandas picks row group size automatically
df.to_parquet("data.parquet")

# explicit row group size (in number of rows)
df.to_parquet("data.parquet", row_group_size=500)


# --- Reading metadata only ---
metadata = pq.read_metadata("data.parquet")
print(metadata.num_rows)                      # total rows
print(metadata.num_row_groups)                # number of row groups
print(metadata.row_group(0).num_rows)         # rows in group 0


# --- Opening without loading data ---
pf = pq.ParquetFile("data.parquet")
print(pf.metadata.num_rows)


# --- Reading a specific row group ---
table = pf.read_row_group(0)                  # pyarrow.Table
df = table.to_pandas()                        # convert to DataFrame


# --- Reading only specific columns ---
table = pf.read_row_group(0, columns=["age", "price"])


# --- Reading the whole file ---
table = pq.read_table("data.parquet")
df = table.to_pandas()

# with column filter
table = pq.read_table("data.parquet", columns=["age", "price"])


# --- Schema and column names ---
print(pf.schema_arrow)                        # full schema with dtypes
print(pf.schema_arrow.names)                  # list of column names


# --- Converting Table formats ---
table.to_pandas()                             # DataFrame
table.to_pydict()                             # dict of lists
table.column("price")                         # single ChunkedArray
table.column("price").to_pylist()             # plain Python list
```